# generate-sj-gex-matrix
St Jude Cloud provisions RNAseq as files per sample, named `{BIOSAMPLE}.RNA-Seq.feature-counts.txt`.  
Format seems to be HTSeq, not FeatureCounts.  

## Requirements
- Directory `data/source/gene-expression/sjcloud/FEATURE_COUNTS/` containing many `{BIOSAMPLE}.RNA-Seq.feature-counts.txt` files. Provisioned by SJC on application.
- env `differential-expression.yml`

## Outputs 
`sjc-gex-counts.tsv`

In [ ]:
Sys.setenv(LANGUAGE = "en")
library(DESeq2)
library(dplyr)

FC_DIR <- "../../data/source/gene-expression/sjcloud/FEATURE_COUNTS"
OUTFILE <- "../../data/source/gene-expression/sjcloud/sjc-gex-counts.tsv.gz"

In [ ]:
import_sj_featurecounts <- function(fc_dir){
    fc_suffix <- "\\.RNA-Seq\\.feature-counts\\.txt$"
    fc_files <- list.files(fc_dir, pattern = fc_suffix)
    sample_table <- data.frame(sampleName = sub(fc_suffix, "", fc_files), fileName = fc_files)
    dds <- DESeqDataSetFromHTSeqCount(sample_table, directory = fc_dir, design = ~ 1)
    return(dds)
}
dds <- import_sj_featurecounts(FC_DIR)

In [ ]:
write_gex_matrix <- function(counts, outfile){
    counts %>%
        as.data.frame %>%
        tibble::rownames_to_column("gene") %>%
        data.table::fwrite(outfile, sep = "\t")
}
write_gex_matrix(counts(dds), OUTFILE)

# generate-cbtn-gex-matrix
CBTN provisions RNAseq as RSEM `.genes.results` files on CAVATICA, and as .rds files at https://s3.amazonaws.com/d3b-openaccess-us-east-1-prd-pbta/open-targets/v15/gene-counts-rsem-expected_count-collapsed.rds

## Requirements
- Directory `data/source/gene-expression/openpedcan` containing `gene-counts-rsem-expected_count-collapsed.rds` file. 
- env `differential-expression.yml`

## Outputs 
`openpedcan-gex-expected-counts.tsv`

In [ ]:
RDS_FILE <- "../../data/source/gene-expression/openpedcan/gene-counts-rsem-expected_count-collapsed.rds"
OPC_OUTFILE <- "../../data/source/gene-expression/openpedcan/openpedcan-gex-expected-counts.tsv.gz"

write_gex_matrix(readRDS(RDS_FILE), OPC_OUTFILE)

In [ ]:
cbtn_counts
